# Hybrid Urban-Scene Segmentation - Colab smoke test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sigurius23/Segmentation-Models/blob/main/notebooks/hybrid_eval_colab_smoke_test.ipynb)

This notebook verifies the repository's complete training and inference workflow. It runs both semantic-segmentation baselines - **SegFormer-B2** and **DeepLabV3+-ResNet101** - through training, checkpointing, checkpoint restoration, inference, and metric reporting.

To mirror the project's hybrid-dataset question without downloading the full research datasets, it creates a tiny Cityscapes/Synscapes-inspired proxy:

- a fixed-size training set mixing **real-like** and **synthetic-like** urban scenes;
- a real-like validation and inference set;
- six street-scene classes: sky, building, vegetation, road, vehicle, and traffic sign;
- identical data, seed, optimizer settings, and evaluation for both models;
- downstream mIoU plus ECE and predictive entropy.

This is an end-to-end engineering smoke test, not a scientific benchmark. The full study must use the real datasets, sweep multiple real-to-synthetic ratios, compute pre-training shift metrics, and correlate those scores with downstream performance.

The repository is public, so **Runtime > Run all** needs no GitHub token. For a faster comparison, select **Runtime > Change runtime type > T4 GPU**. CPU also works.

## 1. Clone or locate the project

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Sigurius23/Segmentation-Models.git"
BRANCH = os.environ.get("HYBRID_EVAL_BRANCH", "main")
local_override = os.environ.get("HYBRID_EVAL_REPO_DIR")
REPO_DIR = Path(local_override).resolve() if local_override else Path("/content/Segmentation-Models")


if not local_override and not REPO_DIR.exists():
    try:
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
            capture_output=True,
            text=True,
        )
    except subprocess.CalledProcessError as exc:
        stderr_lines = [line for line in (exc.stderr or "").splitlines() if line.strip()]
        git_detail = stderr_lines[-1] if stderr_lines else f"git exited with status {exc.returncode}"
        raise RuntimeError(
            "Git clone failed. Check REPO_URL, BRANCH, and the Colab runtime's network access. "
            f"Git reported: {git_detail}"
        ) from None

if not (REPO_DIR / "hybrid_eval").is_dir():
    raise RuntimeError(f"No hybrid_eval package found in {REPO_DIR}. Check REPO_URL and BRANCH.")
if not (REPO_DIR / "requirements-ml.txt").is_file():
    raise RuntimeError(
        "This branch does not contain requirements-ml.txt. Commit and push the completed implementation first."
    )

os.chdir(REPO_DIR)
print(f"Project: {REPO_DIR}")
print(f"Branch:  {BRANCH}")

## 2. Prepare the Colab runtime

Colab normally ships with a compatible PyTorch/TorchVision pair. This cell keeps that pair and installs only missing packages. If PyTorch itself is absent, it falls back to the complete ML requirements.

In [ ]:
import importlib.util
import sys

SKIP_INSTALL = os.environ.get("HYBRID_EVAL_SKIP_INSTALL") == "1"
module_to_requirement = {
    "torch": "torch",
    "torchvision": "torchvision",
    "transformers": "transformers>=4.41,<6",
    "numpy": "numpy>=1.26,<3",
    "PIL": "Pillow>=10,<13",
    "pytest": "pytest>=8,<10",
}
missing = [name for name in module_to_requirement if importlib.util.find_spec(name) is None]
if SKIP_INSTALL:
    print("Dependency installation skipped by HYBRID_EVAL_SKIP_INSTALL=1")
elif "torch" in missing or "torchvision" in missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "-r", "requirements-ml.txt"],
        check=True,
    )
elif missing:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *[module_to_requirement[name] for name in missing]],
        check=True,
    )
else:
    print("All smoke-test dependencies are already available.")

import torch  # noqa: E402 - may be installed above
import torchvision  # noqa: E402 - may be installed above
import transformers  # noqa: E402 - may be installed above

DEVICE = os.environ.get("HYBRID_EVAL_DEVICE") or (
    "cuda"
    if torch.cuda.is_available()
    else (
        "mps"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
        else "cpu"
    )
)
print(
    f"torch={torch.__version__}, torchvision={torchvision.__version__}, "
    f"transformers={transformers.__version__}"
)
print(f"Smoke-test device: {DEVICE}")

## 3. Configure the hybrid-data comparison

The total number of training samples stays fixed while the real-to-synthetic ratio changes, matching the project's experimental-control principle. Both models always receive exactly the same split.

In [ ]:
from hybrid_eval.models import PRIMARY_MODEL_NAMES
from hybrid_eval.models.segformer import SEGFORMER_B2_BACKBONE_CHECKPOINT

REAL_FRACTION = 0.5  # @param {type:"slider", min:0.25, max:0.75, step:0.25}
EPOCHS = 1  # @param {type:"integer", min:1, max:3}
USE_PRETRAINED = False  # @param {type:"boolean"}

REAL_FRACTION = float(os.environ.get("HYBRID_EVAL_REAL_FRACTION", REAL_FRACTION))
EPOCHS = int(os.environ.get("HYBRID_EVAL_SMOKE_EPOCHS", EPOCHS))
USE_PRETRAINED = os.environ.get(
    "HYBRID_EVAL_USE_PRETRAINED", str(USE_PRETRAINED)
).lower() in {"1", "true", "yes"}
requested_models = os.environ.get("HYBRID_EVAL_SMOKE_MODELS", "segformer,deeplabv3plus")
MODELS = tuple(name.strip() for name in requested_models.split(",") if name.strip())
DISPLAY_NAMES = {"segformer": "SegFormer-B2", "deeplabv3plus": "DeepLabV3+-ResNet101"}

if not 0.0 < REAL_FRACTION < 1.0:
    raise ValueError("REAL_FRACTION must be strictly between 0 and 1 for a hybrid split")
if set(MODELS) != set(PRIMARY_MODEL_NAMES):
    raise ValueError(f"This comparison must run {PRIMARY_MODEL_NAMES}")
if "cityscapes" in SEGFORMER_B2_BACKBONE_CHECKPOINT.lower():
    raise AssertionError("SegFormer initialization leaks the Cityscapes target domain")
if EPOCHS < 1:
    raise ValueError("EPOCHS must be positive")

CLASS_NAMES = ("sky", "building", "vegetation", "road", "vehicle", "traffic-sign")
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 64
TRAIN_SAMPLES = 8
EVAL_SAMPLES = 2

print(f"Models: {', '.join(DISPLAY_NAMES[name] for name in MODELS)}")
print(f"Hybrid training ratio: {REAL_FRACTION:.0%} real / {1 - REAL_FRACTION:.0%} synthetic")
print(f"Fixed training size: {TRAIN_SAMPLES}; real-like validation/test: {EVAL_SAMPLES} each")
print(f"Classes ({NUM_CLASSES}): {', '.join(CLASS_NAMES)}")
print(f"Pretrained weights: {USE_PRETRAINED} (ImageNet-only encoders)")

## 4. Create a tiny hybrid urban-scene dataset

The label geometry is shared across domains, while the image appearance changes. Real-like samples use muted colors, texture, illumination gradients, and camera noise. Synthetic-like samples are sharper, more saturated, and contain a mild rendering-stripe artifact. This produces a controlled proxy for the reality gap while keeping perfect labels.

In [ ]:
import shutil

import numpy as np
from PIL import Image, ImageFilter

SMOKE_ROOT = Path(os.environ.get("HYBRID_EVAL_SMOKE_ROOT", "/content/hybrid-eval-smoke"))
if SMOKE_ROOT.exists():
    shutil.rmtree(SMOKE_ROOT)

real_palette = np.array(
    [
        [125, 165, 190],  # sky
        [120, 105, 100],  # building
        [70, 115, 72],  # vegetation
        [75, 75, 78],  # road
        [145, 55, 45],  # vehicle
        [190, 165, 45],  # traffic sign
    ],
    dtype=np.int16,
)
synthetic_palette = np.array(
    [
        [75, 175, 245],
        [165, 95, 130],
        [45, 180, 70],
        [65, 60, 85],
        [235, 45, 65],
        [255, 215, 35],
    ],
    dtype=np.int16,
)


def render_scene(domain, index):
    rng = np.random.default_rng(1000 + index + (0 if domain == "real" else 100))
    height = width = IMAGE_SIZE
    horizon = 21 + index % 3
    road_start = 40 + index % 2
    mask = np.zeros((height, width), dtype=np.uint8)

    mask[horizon:road_start, :] = 1
    vegetation_start = 45 + index % 4
    mask[horizon - 2 : road_start, vegetation_start:] = 2
    mask[road_start:, :] = 3

    vehicle_x = 24 + index % 7
    mask[47:57, vehicle_x : vehicle_x + 17] = 4
    mask[44:48, vehicle_x + 3 : vehicle_x + 13] = 4

    sign_x = 8 + (index * 5) % 15
    mask[29:43, sign_x + 2 : sign_x + 4] = 5
    mask[25:33, sign_x : sign_x + 7] = 5

    palette = real_palette if domain == "real" else synthetic_palette
    image = palette[mask].astype(np.float32)
    yy, xx = np.mgrid[:height, :width]

    if domain == "real":
        illumination = (xx / width) * 16 - (yy / height) * 9
        image += illumination[..., None]
        image += rng.normal(0, 8, size=image.shape)
        pil_image = Image.fromarray(np.clip(image, 0, 255).astype(np.uint8))
        image = np.array(pil_image.filter(ImageFilter.GaussianBlur(radius=0.35)))
    else:
        image += ((xx % 9) == 0)[..., None] * 12
        image += rng.normal(0, 2, size=image.shape)
        image = np.clip(image, 0, 255).astype(np.uint8)

    return image, mask


def write_split(name, domains):
    image_dir = SMOKE_ROOT / "data" / name / "images"
    mask_dir = SMOKE_ROOT / "data" / name / "masks"
    image_dir.mkdir(parents=True)
    mask_dir.mkdir(parents=True)
    for index, domain in enumerate(domains):
        image, mask = render_scene(domain, index)
        filename = f"{domain}_{index:02d}.png"
        Image.fromarray(image).save(image_dir / filename)
        Image.fromarray(mask).save(mask_dir / filename)
    return image_dir, mask_dir


real_count = int(round(TRAIN_SAMPLES * REAL_FRACTION))
synthetic_count = TRAIN_SAMPLES - real_count
train_domains = ["real"] * real_count + ["synthetic"] * synthetic_count
np.random.default_rng(7).shuffle(train_domains)

train_images, train_masks = write_split("train", train_domains)
val_images, val_masks = write_split("val", ["real"] * EVAL_SAMPLES)
test_images, test_masks = write_split("test", ["real"] * EVAL_SAMPLES)

print(f"Hybrid dataset created under {SMOKE_ROOT / 'data'}")
print(f"Training composition: {real_count} real-like + {synthetic_count} synthetic-like")

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

    class_colors = [color / 255 for color in synthetic_palette]
    class_cmap = ListedColormap(class_colors)
    real_image, real_mask = render_scene("real", 0)
    synthetic_image, synthetic_mask = render_scene("synthetic", 0)
    figure, axes = plt.subplots(2, 2, figsize=(8, 7))
    for row, (image, mask, domain) in enumerate(
        [(real_image, real_mask, "Real-like"), (synthetic_image, synthetic_mask, "Synthetic-like")]
    ):
        axes[row, 0].imshow(image)
        axes[row, 0].set_title(f"{domain} image")
        axes[row, 1].imshow(mask, cmap=class_cmap, vmin=0, vmax=NUM_CLASSES - 1)
        axes[row, 1].set_title("Shared semantic labels")
        axes[row, 0].axis("off")
        axes[row, 1].axis("off")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib is unavailable; dataset generation still succeeded.")

## 5. Train both required baselines

Both subprocesses use the same hybrid split and hyperparameters. `--no-pretrained` is used by default so the smoke test does not depend on external model-weight downloads; enable `USE_PRETRAINED` above to load the domain-neutral ImageNet encoders used by the real experiment.

In [ ]:
run_env = os.environ.copy()
run_env["PYTHONPATH"] = str(REPO_DIR)
TRAIN_OUTPUTS = {}

for model_name in MODELS:
    output_dir = SMOKE_ROOT / "training-output" / model_name
    train_command = [
        sys.executable,
        "-m",
        "hybrid_eval.training.train",
        "--model",
        model_name,
        "--train-img-dir",
        str(train_images),
        "--train-mask-dir",
        str(train_masks),
        "--val-img-dir",
        str(val_images),
        "--val-mask-dir",
        str(val_masks),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        "2",
        "--backbone-lr",
        "0.001",
        "--head-lr",
        "0.001",
        "--image-height",
        str(IMAGE_SIZE),
        "--image-width",
        str(IMAGE_SIZE),
        "--crop-size",
        str(IMAGE_SIZE),
        "--num-classes",
        str(NUM_CLASSES),
        "--num-workers",
        "0",
        "--output-dir",
        str(output_dir),
        "--compact-checkpoints",
        "--device",
        DEVICE,
        "--seed",
        "7",
    ]
    if not USE_PRETRAINED:
        train_command.append("--no-pretrained")

    print(f"\nTraining {model_name}: {' '.join(train_command)}")
    subprocess.run(train_command, cwd=REPO_DIR, env=run_env, check=True)
    TRAIN_OUTPUTS[model_name] = output_dir
    print(f"Training completed: {model_name}")

## 6. Restore both checkpoints and run standalone inference

In [ ]:
CHECKPOINTS = {}
INFERENCE_OUTPUTS = {}

for model_name in MODELS:
    checkpoint = TRAIN_OUTPUTS[model_name] / f"best_model_{model_name}.pth"
    inference_output = SMOKE_ROOT / "inference-output" / model_name
    inference_command = [
        sys.executable,
        "-m",
        "hybrid_eval.inference",
        "--checkpoint",
        str(checkpoint),
        "--image-dir",
        str(test_images),
        "--mask-dir",
        str(test_masks),
        "--output-dir",
        str(inference_output),
        "--batch-size",
        "2",
        "--num-workers",
        "0",
        "--device",
        DEVICE,
    ]

    print(f"\nInference for {model_name}: {' '.join(inference_command)}")
    subprocess.run(inference_command, cwd=REPO_DIR, env=run_env, check=True)
    CHECKPOINTS[model_name] = checkpoint
    INFERENCE_OUTPUTS[model_name] = inference_output
    print(f"Inference completed: {model_name}")

## 7. Compare downstream performance and uncertainty

mIoU is higher-is-better. ECE and predictive entropy are lower-is-better here, but a one-epoch random-initialization smoke run is not suitable for architecture ranking.

In [ ]:
import json

comparison = []
prediction_paths = {}
for model_name in MODELS:
    history_path = TRAIN_OUTPUTS[model_name] / f"training_history_{model_name}.json"
    summary_path = INFERENCE_OUTPUTS[model_name] / "summary.json"
    history = json.loads(history_path.read_text())
    summary = json.loads(summary_path.read_text())
    masks = sorted((INFERENCE_OUTPUTS[model_name] / "masks").glob("*.png"))

    assert CHECKPOINTS[model_name].is_file(), f"Missing checkpoint for {model_name}"
    assert len(history) == EPOCHS, f"Unexpected training history for {model_name}"
    assert summary["model"] == model_name
    assert summary["images"] == EVAL_SAMPLES
    assert len(masks) == EVAL_SAMPLES
    assert summary["metrics"]["valid_pixels"] == EVAL_SAMPLES * IMAGE_SIZE * IMAGE_SIZE
    assert 0.0 <= summary["metrics"]["mIoU"] <= 1.0

    final_epoch = history[-1]
    metrics = summary["metrics"]
    comparison.append(
        {
            "model": model_name,
            "validation_mIoU": final_epoch["val_mIoU"],
            "test_mIoU": metrics["mIoU"],
            "test_ECE": metrics["ece"],
            "test_entropy": metrics["entropy"],
            "prediction_masks": len(masks),
        }
    )
    prediction_paths[model_name] = masks

comparison_path = SMOKE_ROOT / "model_comparison.json"
comparison_path.write_text(json.dumps(comparison, indent=2))

print("\nProject-specific model comparison")
print("model       | val mIoU | test mIoU | test ECE | entropy | masks")
print("------------|----------|-----------|----------|---------|------")
for row in comparison:
    print(
        f"{row['model']:<11} | {row['validation_mIoU']:.4f}   | "
        f"{row['test_mIoU']:.4f}    | {row['test_ECE']:.4f}   | "
        f"{row['test_entropy']:.4f}  | {row['prediction_masks']}"
    )

print(f"\nComparison JSON: {comparison_path}")
print("BOTH-MODEL SMOKE TEST PASSED")

In [ ]:
try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

    class_cmap = ListedColormap([color / 255 for color in synthetic_palette])
    sample_name = "real_00.png"
    source_image = np.array(Image.open(test_images / sample_name))
    target_mask = np.array(Image.open(test_masks / sample_name))

    figure, axes = plt.subplots(1, 2 + len(MODELS), figsize=(4 * (2 + len(MODELS)), 3.8))
    axes[0].imshow(source_image)
    axes[0].set_title("Real-like test image")
    axes[1].imshow(target_mask, cmap=class_cmap, vmin=0, vmax=NUM_CLASSES - 1)
    axes[1].set_title("Target")
    for index, model_name in enumerate(MODELS, start=2):
        predicted_mask = np.array(Image.open(INFERENCE_OUTPUTS[model_name] / "masks" / sample_name))
        axes[index].imshow(predicted_mask, cmap=class_cmap, vmin=0, vmax=NUM_CLASSES - 1)
        axes[index].set_title(DISPLAY_NAMES[model_name])
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()

    figure, axes = plt.subplots(1, 3, figsize=(12, 3.5))
    model_labels = [DISPLAY_NAMES[row["model"]] for row in comparison]
    for axis, key, title in [
        (axes[0], "test_mIoU", "Test mIoU (higher is better)"),
        (axes[1], "test_ECE", "Test ECE (lower is better)"),
        (axes[2], "test_entropy", "Predictive entropy (lower is better)"),
    ]:
        axis.bar(model_labels, [row[key] for row in comparison], color=["#4C78A8", "#F58518"])
        axis.set_title(title)
        axis.set_ylim(bottom=0)
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib is unavailable; artifact and metric verification still passed.")

## 8. Run focused regression tests

These tests cover paired image/mask loading, augmentation alignment, streaming mIoU/ECE/entropy, train/validation steps, checkpoint handling, and standalone inference behavior.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_hybrid_eval.py", "-q", "-p", "no:cacheprovider"],
    cwd=REPO_DIR,
    env=run_env,
    check=True,
)
print("Focused regression tests passed.")

## What this establishes

- Both required model loaders construct a runnable segmentation architecture.
- Both pretrained paths use domain-neutral ImageNet encoders rather than Cityscapes-fine-tuned weights.
- Both models train through the same CLI on a fixed-size hybrid dataset.
- Both write self-describing checkpoints and reload them in a separate inference process.
- Both produce segmentation masks and downstream mIoU, ECE, and predictive-entropy results.
- The focused regression suite passes in the same runtime.

For the actual research experiment, replace the proxy images with aligned Cityscapes/Synscapes/GTA5-style datasets, sweep the planned real-to-synthetic ratios while holding sample count constant, compute the pre-training distribution-shift metrics, and evaluate their correlation with real-domain downstream performance.